In [1]:
import logging
from pathlib import Path

from nullvector import NullVectorClient
from nullvector.observability import configure_default_runtime_observability

# Configure structured JSONL and console progress logging
logger = configure_default_runtime_observability(level=logging.INFO)

# Initialize the client with a local filesystem workspace
WORKSPACE_DIR = Path("./nv_workspace")
WORKSPACE_DIR.mkdir(exist_ok=True)

client = NullVectorClient(storage_path=WORKSPACE_DIR, logger=logger)

In [3]:
from nullvector import SourceDocumentKind

source_file = "/home/pruthvi/projects/NullVector/cookbook/903000608.pdf" # Replace with your target PDF or Markdown file

# Acquire returns the manifest and its storage reference path
manifest, manifest_ref = client.acquire(
    source_path=source_file,
    source_kind=SourceDocumentKind.PDF,
    preset="academic_paper"  # Presets tune layout heuristics (e.g., academic_paper, financial_report)
)

print(f"Document ID (SHA256): {manifest.document_id}")
print(f"Total Pages Parsed: {manifest.page_count}")
print(f"Ledger Artifact Path: {manifest.ledger_path}")

[SourceFingerprintComputed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896


[AcquisitionStarted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 run=903000608-acquisition provider=native_pymupdf
'utf-16-be' codec can't decode byte 0x31 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41'
'utf-16-be' codec can't decode byte 0x44 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD'
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD' from destination
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41' from destination
[PageNativeParsed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 blocks=4
[PageProfiled] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 unresolved=0
[PageNativeParsed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 blocks=56
[PageProfiled] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 unresolve

Document ID (SHA256): 798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
Total Pages Parsed: 148
Ledger Artifact Path: /home/pruthvi/projects/NullVector/cookbook/nv_workspace/903000608-acquisition/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/ledger/canonical-document-ledger.json


In [4]:
from nullvector import CanonicalDocumentLedger
from nullvector._client_utils import load_model_artifact

# Load the persisted ledger artifact
ledger = load_model_artifact(
    CanonicalDocumentLedger,
    Path(manifest_ref).parent / manifest.ledger_path
)

# Inspect blocks on the first parsed page
first_page = ledger.pages[0]
print(f"Page 1 Dimensions: {first_page.width}x{first_page.height}")
print(f"Total Blocks Found: {len(first_page.blocks)}")

# Extract and print the first text block identified
for block in first_page.blocks:
    if block.block_type == "text_block":
        print(f"\n--- First Text Block (Reading Index: {block.reading_index}) ---")
        print(f"Bounding Box: {block.bbox}")
        print(block.content[:300] + "...\n")
        break

Page 1 Dimensions: 585.56396484375x809.4010009765625
Total Blocks Found: 4


In [5]:
from nullvector import AcquisitionRequest, AcquisitionSettings

custom_settings = AcquisitionSettings(
    detect_tables=True,
    table_min_columns=3,
    image_region_warning_threshold=0.75, # Flag pages that are mostly images
    render_dpi=144
)

custom_request = AcquisitionRequest(
    source_path=source_file,
    acquisition_run_id="custom-acquisition-run-01",
    artifact_root=str(WORKSPACE_DIR),
    source_kind=SourceDocumentKind.PDF,
    provider_identity="native_pymupdf",
    settings=custom_settings
)

# You can pass this request directly to the underlying AcquisitionService
from nullvector.ingest import AcquisitionService

service = AcquisitionService(logger=logger)

custom_manifest = service.acquire(custom_request)
print(f"Custom acquisition completed. Artifacts at: {custom_manifest.artifact_root}")

[SourceFingerprintComputed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
[AcquisitionStarted] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 run=custom-acquisition-run-01 provider=native_pymupdf
'utf-16-be' codec can't decode byte 0x31 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41'
'utf-16-be' codec can't decode byte 0x44 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD'
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD' from destination
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41' from destination
[PageNativeParsed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 blocks=4
[PageProfiled] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 unresolved=0
[PageNativeParsed] document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 bl

Custom acquisition completed. Artifacts at: nv_workspace/custom-acquisition-run-01/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896


In [ ]:
import os

from openai import OpenAI

from nullvector import NullVectorClient
from nullvector.llm import GatewayConfig, GatewayService, StructuredOutputMode
from nullvector.llm.adapters import OpenAIAdapter

# 1. Initialize the underlying provider SDK
# (Ensure you have your OPENAI_API_KEY set in your environment)
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
adapter = OpenAIAdapter(client=openai_client)

# 2. Configure the NullVector Gateway
# The gateway adds circuit breaking, retries, and strict schema validation
gateway_config = GatewayConfig(
    default_model="gpt-4o-mini", # Or "gpt-4o" for better multimodal visual tasks
    supported_structured_output_modes=(StructuredOutputMode.PROVIDER_NATIVE,)
)
gateway = GatewayService(config=gateway_config, provider_adapter=adapter, logger=logger)

# 3. Re-initialize the client with the gateway attached
client = NullVectorClient(
    storage_path=WORKSPACE_DIR,
    gateway=gateway,
    logger=logger
)

print("✅ LLM Gateway successfully attached to the NullVector client.")

In [ ]:
# We assume `manifest_ref` is still in memory from the Phase 01 acquisition cell.
# If not, you can locate it in the workspace catalog.

print("Building hierarchy tree and generating semantic summaries...")

tree_manifest = client.build_tree(
    acquisition_manifest_path=manifest_ref,
    preset="academic_paper",
    summarize=True # Triggers the NodeSummarizer via the LLM Gateway
)

print(f"✅ Tree Build Complete: {tree_manifest.tree_run_id}")
print(f"Committed Nodes: {tree_manifest.committed_node_count}")
print(f"Tree Artifacts saved at: {tree_manifest.artifact_root}")

In [ ]:
# Build the indexable retrieval corpus
retrieval_manifest = client.build_retrieval(
    acquisition_manifest_path=manifest_ref,
    tree_manifest_path=tree_manifest.manifest_path # Optional, but enables tree-guided retrieval
)

print(f"✅ Retrieval Corpus Built: {retrieval_manifest.corpus_path}")
print(f"Total Retrieval Units Indexed: {retrieval_manifest.unit_count}")

In [ ]:
# Ask a question that requires visual interpretation or complex semantic synthesis
query = "Explain the workflow depicted in the architecture diagram in the methodology section."

print(f"Asking: '{query}'\n")

# The ask() method handles query planning, ranking, and answer synthesis
qa_response = client.ask(
    query=query,
    document_id=manifest.document_id # Resolves the latest retrieval corpus via the local catalog
)

print("--- 🤖 Answer ---")
print(qa_response.answer)
print("\n--- 📚 Grounding & Citations ---")
print(f"Answer Strategy Used: {qa_response.answer_strategy}")

# Inspect the exact evidence used to ground the answer
for i, citation in enumerate(qa_response.citations, 1):
    print(f"\nCitation {i}:")
    print(f"  Page: {citation.page_label}")
    if citation.quote:
        print(f"  Excerpt: '{citation.quote[:150]}...'")
    if citation.asset_path:
        print(f"  🖼️ Visual Evidence Path: {citation.asset_path}")